## Analyst lesson 4.2 — Value the same borrower from native UFCF


In [ ]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
from _shared.borrower_model import borrower_model, borrower_vintage
from finstack_quant.core.market_data import MarketContext
from finstack_quant.core.money import Money
from finstack_quant.statements import ModelBuilder, Evaluator, ForecastSpec, MonteCarloConfig
from finstack_quant.statements_analytics import (
    CorporateDcfRequest,
    DcfInputs,
    DcfOptions,
    DcfProjection,
    DcfSensitivityGridSpec,
    StatementValuationCut,
    TerminalFcffBasis,
    TerminalValueSpec,
    WaccInputs,
    dcf_sensitivity_grid,
    evaluate_corporate_dcf,
    evaluate_dcf_from_results,
    evaluate_dcf_monte_carlo,
    dcf_sensitivity_on_results,
    wacc,
    ScenarioSet,
    evaluate_scenario_set,
    SensitivityConfig,
    run_sensitivity,
    generate_tornado_entries,
    goal_seek,
    variance_bridge,
)

borrower = borrower_model()
PERIODS = tuple(borrower.periods)
borrower_results = Evaluator().evaluate(borrower)
valuation_builder = borrower.to_builder()
# Expose operating growth as native formula drivers. Preserve every actual and the baseline forecast.
for node, growth in [("revenue", 0.02), ("cogs", 0.02), ("opex", 0.015)]:
    valuation_builder.value(f"{node}_growth", [(period, growth) for period in PERIODS])
    valuation_builder.remove_node(node)
    valuation_builder = (
        valuation_builder
        .mixed(node)
        .values([(period, borrower_results.get(node, period)) for period in PERIODS[:4]])
        .formula(f"lag({node},1) * (1 + {node}_growth)")
        .build()
    )
# The operating graph stores currency-unit scalars. Stamp USD once at the DCF boundary.
valuation_builder.currency("USD")
valuation_builder.as_money("ufcf_usd", "ufcf")
valuation_builder.as_money("ebitda_usd", "ebitda")
valuation_builder.as_money("total_debt", "debt_end").as_money("cash", "cash_end")
AS_OF = "2025-01-01"
TERMINAL_DATE = "2025-12-31"
market = MarketContext()


def dated_model(builder):
    """Record the publication cut for the illustrative borrower snapshot."""
    builder.source_snapshot("borrower-2024Q4-published")
    snapshot = builder.build()
    builder = snapshot.to_builder()
    for node in snapshot.nodes:
        actuals = [period for period in (node.values or {}) if period in PERIODS[:4]]
        if actuals:
            builder.availability_dates(node.node_id, [(period, AS_OF) for period in actuals])
    return builder.build()


valuation_model = dated_model(valuation_builder)
valuation_results = Evaluator().evaluate_with_market(valuation_model, market, AS_OF, AS_OF)
assert all(abs(valuation_results.get("ufcf", p) - borrower_results.get("ufcf", p)) < 1e-6 for p in PERIODS)
cost_of_capital = wacc(
    WaccInputs(equity_weight="0.65", cost_of_equity=0.12, debt_weight="0.35", cost_of_debt=0.075, tax_rate=0.25)
)
assert abs(cost_of_capital - (0.65 * 0.12 + 0.35 * 0.075 * (1 - 0.25))) < 1e-12
terminal_growth = 0.025
terminal = TerminalValueSpec.gordon_growth(terminal_growth)
cut = StatementValuationCut(AS_OF, AS_OF, "USD")


def valuation_request(terms, projection, options=None):
    return CorporateDcfRequest(
        cut=cut,
        inputs=DcfInputs(wacc=cost_of_capital, terminal_value=terms, ufcf_node="ufcf_usd"),
        projection=projection,
        options=options,
    )


growth_projection = DcfProjection(
    terminal_date=TERMINAL_DATE,
    normalized_annual_fcff=TerminalFcffBasis.trailing_additive(),
)
base_request = valuation_request(terminal, growth_projection)
base_dcf = evaluate_dcf_from_results(valuation_model, valuation_results, base_request)
pre_discount_common_equity = (
    base_dcf.enterprise_value.amount - base_dcf.net_debt.amount - base_dcf.other_equity_claims.amount
)
assert abs(pre_discount_common_equity - base_dcf.valuation_discount.amount - base_dcf.equity_value.amount) < 0.01
assert abs(base_dcf.net_debt.amount - valuation_results.get("net_debt", "2024Q4")) < 0.01
terminal_annual_ufcf = sum(valuation_results.get("ufcf", p) for p in PERIODS[4:])
terminal_undiscounted = terminal_annual_ufcf * (1 + terminal_growth) / (cost_of_capital - terminal_growth)
terminal_discount_factor = base_dcf.terminal_value_pv.amount / terminal_undiscounted
explicit_pv = base_dcf.enterprise_value.amount - base_dcf.terminal_value_pv.amount
print("Forecast quarterly UFCF (USD):", [valuation_results.get("ufcf", p) for p in PERIODS[4:]])
print(f"WACC={cost_of_capital:.5%}; annual terminal UFCF={terminal_annual_ufcf:,.2f}")
print(base_dcf.to_json())
print(f"Terminal share of enterprise value={base_dcf.terminal_value_pv.amount / base_dcf.enterprise_value.amount:.2%}")

## Analyst lesson 4.2 — Native DCF sensitivity and an operating tornado


In [ ]:
from finstack_quant.reporting.charts import tornado_chart
from finstack_quant.reporting.theme import INSTITUTIONAL
from IPython.display import SVG, display

axes = DcfSensitivityGridSpec(
    [0.0876875, 0.1076875],
    [0.015, 0.025, 0.035],
)
grid = dcf_sensitivity_grid(valuation_model, valuation_results, base_request, axes).to_dataframe()
assert grid.shape == (2, 3)
assert (grid.diff(axis=0).iloc[1:] < 0).all().all()
assert (grid.diff(axis=1).iloc[:, 1:] > 0).all().all()
print(grid.to_string())
dcf_shocks = dcf_sensitivity_on_results(
    valuation_model,
    valuation_results,
    base_request,
    {
        "wacc": {"low": "0.01", "high": "0.0125"},
        "terminal": {"parameter": "stable_growth", "shocks": {"low": "0.005", "high": "0.01"}},
    },
)
rows = []
for pair in dcf_shocks["pairs"]:
    low, high = pair["low"], pair["high"]
    assert low["status"] == high["status"] == "priced"
    rows.append((pair["parameter_id"], float(low["change"]["amount"]), float(high["change"]["amount"])))
print("Native EV deltas under lower/higher parameter shocks:", pd.DataFrame(rows, columns=["parameter", "low", "high"]))
dcf_svg = tornado_chart(
    [(parameter, min(low, high) / 1e6, max(low, high) / 1e6) for parameter, low, high in rows],
    theme=INSTITUTIONAL,
)
Path("dcf-tornado-usd-millions.svg").write_text(dcf_svg)
display(SVG(dcf_svg))
operating_sensitivity = run_sensitivity(
    valuation_model,
    SensitivityConfig(
        mode="tornado",
        parameters=[
            ("revenue_growth", "2025Q4", 0.02, [-0.02, 0.06]),
            ("opex_growth", "2025Q4", 0.015, [-0.015, 0.045]),
        ],
        target_metrics=["ufcf"],
    ),
)
operating_entries = generate_tornado_entries(operating_sensitivity, "ufcf", "2025Q4")
assert len(operating_entries) == 2
print(
    "Operating sensitivity is quarterly UFCF, not EV:",
    [(e.parameter_id, e.downside, e.upside) for e in operating_entries],
)
operating_svg = tornado_chart(
    [(e.parameter_id, min(e.downside, e.upside) / 1e3, max(e.downside, e.upside) / 1e3) for e in operating_entries],
    theme=INSTITUTIONAL,
)
Path("operating-tornado-usd-thousands.svg").write_text(operating_svg)
display(SVG(operating_svg))

## Analyst lesson 4.2 — Price bull/base/bear native statement paths


In [ ]:
scenario_set = ScenarioSet({"base": {}, "bull": {"revenue_growth": 0.04}, "bear": {"revenue_growth": -0.02}})
scenario_results = evaluate_scenario_set(valuation_model, scenario_set)


def price_ufcf_path(forecast_values):
    """Carry one native quarterly UFCF path into the native DCF without mixing marginal percentiles."""
    money_values = [
        (p, Money(valuation_results.get("ufcf", p) if p in PERIODS[:4] else float(forecast_values[p]), "USD"))
        for p in PERIODS
    ]
    priced_builder = valuation_model.to_builder().remove_node("ufcf_usd").value_money("ufcf_usd", money_values)
    return evaluate_corporate_dcf(dated_model(priced_builder), base_request, market)


scenario_rows = []
for name in scenario_results.names:
    result = scenario_results.get(name)
    assert all(result.get("revenue", p) == valuation_results.get("revenue", p) for p in PERIODS[:4])
    value = price_ufcf_path({p: result.get("ufcf", p) for p in PERIODS[4:]})
    scenario_rows.append({
        "scenario": name,
        "2025Q4_UFCF_usd": result.get("ufcf", "2025Q4"),
        "equity_usd": value.equity_value.amount,
    })
scenario_table = pd.DataFrame(scenario_rows).set_index("scenario")
assert (
    scenario_table.loc["bull", "equity_usd"]
    > scenario_table.loc["base", "equity_usd"]
    > scenario_table.loc["bear", "equity_usd"]
)
assert abs(scenario_table.loc["base", "equity_usd"] - base_dcf.equity_value.amount) < 0.01
print(scenario_table.to_string())

## Analyst lesson 4.2 — Solve a price-implied terminal assumption


In [ ]:
goal_builder = valuation_model.to_builder()
goal_builder.value("wacc_assumption", [(p, cost_of_capital) for p in PERIODS])
goal_builder.value("terminal_growth_assumption", [(p, terminal_growth) for p in PERIODS])
goal_builder.compute("ufcf_ttm", "ufcf + lag(ufcf,1) + lag(ufcf,2) + lag(ufcf,3)")
goal_builder.as_money(
    "terminal_ev_hook", "ufcf_ttm * (1+terminal_growth_assumption)/(wacc_assumption-terminal_growth_assumption)"
)
goal_model = goal_builder.build()
market_equity = 130_000_000.0
market_enterprise = market_equity + base_dcf.net_debt.amount + base_dcf.other_equity_claims.amount
required_terminal = (market_enterprise - explicit_pv) / terminal_discount_factor
implied = goal_seek(
    goal_model,
    {
        "target_node": "terminal_ev_hook",
        "target_period": "2025Q4",
        "target_value": required_terminal,
        "driver_node": "terminal_growth_assumption",
        "driver_period": "2025Q4",
        "update_model": True,
        "bounds": (-0.02, 0.045),
    },
)
implied_request = valuation_request(TerminalValueSpec.gordon_growth(implied.solved_value), growth_projection)
repriced = evaluate_dcf_from_results(valuation_model, valuation_results, implied_request)
assert abs(repriced.equity_value.amount - market_equity) < 0.01
print(
    f"Target equity={market_equity:,.2f}; implied perpetual growth={implied.solved_value:.6%}; native reprice={repriced.equity_value.amount:,.2f}"
)

## Analyst lesson 4.2 — Reconcile Gordon growth with an exit multiple


In [ ]:
exit_ebitda = sum(valuation_results.get("ebitda", p) for p in PERIODS[4:])
equivalent_multiple = terminal_undiscounted / exit_ebitda
exit_request = valuation_request(
    TerminalValueSpec.exit_multiple(equivalent_multiple),
    DcfProjection(terminal_date=TERMINAL_DATE),
    DcfOptions(exit_multiple_metric_node="ebitda_usd"),
)
exit_value = evaluate_dcf_from_results(valuation_model, valuation_results, exit_request)
assert abs(exit_value.enterprise_value.amount - base_dcf.enterprise_value.amount) < 0.01
print(f"Equivalent exit EV / trailing-year EBITDA={equivalent_multiple:.6f}x")
print(f"Gordon EV={base_dcf.enterprise_value.amount:,.2f}; exit-multiple EV={exit_value.enterprise_value.amount:,.2f}")
print(
    "The multiple is implied by this cash-conversion ratio, discount rate and growth; it is not an independent market comparable."
)

## Analyst lesson 4.2 — Explain revised UFCF with additive signed components


In [ ]:
vintage_results = []
for vintage in ("2024Q3", "2024Q4"):
    builder = borrower_vintage(vintage).to_builder()
    builder.compute("after_tax_ebit", "ebit * (1-tax_rate)")
    builder.compute("capex_outflow", "-capex").compute("nwc_outflow", "-change_nwc")
    vintage_results.append(Evaluator().evaluate(builder.build()))
vintage_bridge = variance_bridge(
    vintage_results[0],
    vintage_results[1],
    "ufcf",
    "2025Q4",
    ["after_tax_ebit", "depreciation", "capex_outflow", "nwc_outflow"],
    "Q3 information",
    "Q4 information",
)
assert abs(vintage_bridge.unexplained) < 1e-6
print(vintage_bridge.to_json())
print(
    "Raw driver deltas reconcile because these nodes are additive signed UFCF components. Revenue alone is not a causal EV attribution."
)

## Analyst lesson 4.2 — Value retained joint statement paths


In [ ]:
import matplotlib.pyplot as plt

stochastic_model = (
    valuation_model.to_builder().forecast("revenue", ForecastSpec.log_normal(math.log(1.02), 0.025, 17)).build()
)
mc_config = MonteCarloConfig(1024, 17, ["ufcf_usd"], [0.05, 0.5, 0.95], True)
statement_mc = Evaluator().evaluate_monte_carlo(
    stochastic_model, mc_config, market=market, as_of=AS_OF, observation_cutoff=AS_OF
)
stochastic_baseline = Evaluator().evaluate_with_market(stochastic_model, market, AS_OF, AS_OF)
distribution = evaluate_dcf_monte_carlo(stochastic_model, stochastic_baseline, statement_mc, base_request)
path_values = distribution.to_dataframe()
assert len(path_values) == 1024 and path_values["path_id"].is_unique
enterprise_values = path_values["enterprise_value"].to_numpy()
assert np.isfinite(enterprise_values).all() and enterprise_values.std(ddof=1) > 0
print(
    "Native statement paths / native DCF EV p5,p50,p95 (USD):",
    distribution.quantiles_to_dataframe()["enterprise_value"].to_list(),
)
print(
    "Distribution of modeled outcomes, not a confidence interval for a known fair value. WACC, terminal growth and costs are fixed."
)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.hist(enterprise_values / 1e6, bins=35, color="#167d77", edgecolor="white")
ax.axvline(base_dcf.enterprise_value.amount / 1e6, color="#a45227", label="Deterministic base")
ax.set(
    xlabel="Enterprise value (USD millions)",
    ylabel="Simulated paths",
    title="Revenue-growth uncertainty through the borrower UFCF graph",
)
ax.legend()
fig.tight_layout()